# Deploy Recommendation Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **5-agent e-commerce assistant** using **Strands Agents GraphBuilder** for deterministic routing across Amazon Bedrock AgentCore runtimes. In this notebook, you deploy the Recommendation Agent -- a pure LLM synthesis agent that generates personalized product recommendations.

**Notebook 4 of 5** -- The Graph Orchestrator routes to this agent only for RECOMMEND intents, after Product and Order agents complete in parallel.

## Architecture Overview

This tutorial deploys 5 agents across 5 Amazon Bedrock AgentCore runtimes. The Recommendation Agent (highlighted below) is the final node in the RECOMMEND path:

| Runtime | Agent | Protocol | Tools |
|---------|-------|----------|-------|
| 1 | Classifier | A2A (port 9000) | None (pure LLM) |
| 2 | Product | A2A (port 9000) | HTTP API tools |
| 3 | Order | A2A (port 9000) | DynamoDB via MCP |
| **4** | **Recommendation** | **A2A (port 9000)** | **None (LLM synthesis)** |
| 5 | Graph Orchestrator | HTTP (port 8080) | GraphBuilder + A2A clients |

The Recommendation Agent is invoked only for **RECOMMEND** intent:
- Receives product catalog data from the Product Agent
- Receives order history from the Order Agent
- Generates 3-5 personalized recommendations by cross-referencing both datasets

GraphBuilder's `_build_node_input()` automatically passes all predecessor outputs as context.

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- **Notebooks 01-03 completed**: Classifier, Product, and Order agents deployed

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

NOTEBOOK_DIR = Path.cwd()

from utils import (
    RECOMMENDATION_AGENT_NAME,
    RECOMMENDATION_ROLE_NAME,
    SSM_RECOMMENDATION_AGENT_URL,
    create_agentcore_role,
    store_agent_url,
)

session = boto3.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Agent Name: {RECOMMENDATION_AGENT_NAME}")

---
## Step 1: Create Recommendation Agent with A2A Protocol

The Recommendation Agent is a **pure LLM synthesis agent** with no tools. In the graph execution flow, GraphBuilder automatically passes the outputs from completed predecessor nodes (Product and Order) as context to this agent via `_build_node_input()`.

The agent then cross-references product catalog data with customer order history to generate personalized recommendations.

### Code Structure

The following cell creates `recommendation_agent/a2a_server.py`:

| Section | What It Does |
|---------|-------------|
| System Prompt | Instructs the agent to cross-reference product and order data for recommendations |
| Agent Creation | Pure LLM agent with no tools -- synthesis via prompt engineering |
| A2A Server | Wraps agent for inter-agent communication on port 9000 |
| Health Check | `/ping` endpoint for AgentCore container monitoring |

In [ ]:
%%writefile recommendation_agent/a2a_server.py
"""Recommendation Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Synthesizes personalized product recommendations using context from graph predecessor
nodes. Receives product catalog data and order history via GraphBuilder's
_build_node_input() and generates tailored suggestions without any external tool calls.

Key features:
- A2A protocol server for graph-based multi-agent orchestration via Agent-to-Agent messaging
- Pure LLM synthesis with no tools -- all reasoning from predecessor node context
- OpenTelemetry instrumentation for AWS X-Ray distributed tracing
"""

import json
import logging
import os

import boto3
import uvicorn
from fastapi import FastAPI
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

StrandsTelemetry().setup_otlp_exporter()

PORT = 9000
SSM_RECOMMENDATION_AGENT_URL = "/ecommerce-graph/recommendation-agent-url"

SYSTEM_PROMPT = """You are a Recommendation Agent for an e-commerce assistant.

You receive context from upstream agents in a graph-based orchestration pipeline:
- Product catalog data (available products with names, categories, prices, ratings)
- Customer order history (past purchases with dates, quantities, totals)

Your task is to generate personalized product recommendations by cross-referencing
the customer's purchase patterns and browsing interests with the available catalog.

Instructions:
1. Analyze the customer's purchase patterns from their order history.
2. Cross-reference past purchases with available products in the catalog.
3. Generate 3-5 personalized product recommendations with clear explanations.
4. For each recommendation, include:
   - Product name
   - Category
   - Why it is recommended (based on purchase history or product attributes)
   - Price, if available from the catalog data
5. If you receive only product data (no order history), recommend popular or
   highly-rated products from the catalog.
6. If you receive only order data (no product catalog), suggest complementary
   products based on past purchase categories and patterns.
7. Keep recommendations concise and conversational.
"""


class ToolLoggingHandler:
    def __init__(self):
        self.logged_tool_ids = set()
        self.tool_count = 0
    def __call__(self, **kwargs):
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_use.get('name', 'Unknown')} ===")
                            input_str = json.dumps(tool_use.get("input", {}))
                            if len(input_str) > 2000:
                                input_str = input_str[:2000] + "..."
                            logger.info(f"TOOL INPUT: {input_str}")
        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# Get AWS region
session = boto3.Session()
region = session.region_name or os.environ.get("AWS_REGION", "us-west-2")

recommendation_agent = Agent(
    name="Ecommerce_Graph_Recommendation",
    description="Personalized product recommendation agent for e-commerce assistant",
    system_prompt=SYSTEM_PROMPT,
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[],
    callback_handler=ToolLoggingHandler(),
)

app = FastAPI()

@app.get("/ping")
async def health():
    return {"status": "healthy"}

a2a_server = A2AServer(agent=recommendation_agent, serve_at_root=True)
a2a_server.setup(app)

if __name__ == "__main__":
    logger.info(f"Starting Recommendation Agent on port {PORT}")
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile recommendation_agent/requirements.txt
strands-agents[a2a,otel]
strands-agents-tools
fastapi
uvicorn
boto3

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline:

1. **Creates IAM role** -- Grants permissions for ECR image pull, Bedrock model invocation, and CloudWatch logging
2. **Configures runtime** -- Packages agent code into a deployable container configuration
3. **Launches runtime** -- Builds Docker image, pushes to ECR, and creates the AgentCore runtime

### Create IAM Role and Configure Runtime

| Parameter | Value | Purpose |
|-----------|-------|--------|
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server |
| `protocol` | `A2A` | Enables Agent-to-Agent communication on port 9000 |
| `agent_name` | `ecommerce_graph_recommendation` | Unique identifier for this runtime |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

recommendation_role_arn = create_agentcore_role(RECOMMENDATION_ROLE_NAME, account_id, region)
print(f"IAM Role ARN: {recommendation_role_arn}")

recommendation_agent_dir = NOTEBOOK_DIR / "recommendation_agent"
os.chdir(recommendation_agent_dir)

recommendation_runtime = Runtime()
recommendation_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=recommendation_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=RECOMMENDATION_AGENT_NAME,
    protocol="A2A",
)

os.chdir(NOTEBOOK_DIR)
print(f"Runtime configured: {RECOMMENDATION_AGENT_NAME}")

### Fix Dockerfile Permissions

AgentCore containers run as the `bedrock_agentcore` user, not root. The auto-generated Dockerfile uses `COPY . .` which preserves host file ownership, causing permission errors. This cell updates it to `COPY --chown=bedrock_agentcore:bedrock_agentcore . .`.

In [ ]:
# # Fix Dockerfile COPY ownership
# dockerfile_path = "Dockerfile"
# with open(dockerfile_path, 'r') as f:
#     content = f.read()
# if 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .' not in content:
#     content = content.replace('COPY . .', 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .')
#     with open(dockerfile_path, 'w') as f:
#         f.write(content)
#     print("Dockerfile updated with correct ownership")
# else:
#     print("Dockerfile already has correct ownership")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime.

**Note:** First deployment takes 5-10 minutes (subsequent updates are faster).

In [ ]:
print("Launching Recommendation Agent (this may take several minutes)...")
os.chdir(recommendation_agent_dir)
recommendation_launch = recommendation_runtime.launch(auto_update_on_conflict=True)
print(f"Recommendation Agent ARN: {recommendation_launch.agent_arn}")
RECOMMENDATION_AGENT_ARN = recommendation_launch.agent_arn
os.chdir(NOTEBOOK_DIR)

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN.

In [ ]:
os.chdir(recommendation_agent_dir)
status_response = recommendation_runtime.status()
status = status_response.endpoint.get("status", "")
recommendation_agent_url = None

print(f"Recommendation Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    recommendation_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Runtime URL: {recommendation_agent_url}")
else:
    print(f"Agent not ready. Current status: {status}")

os.chdir(NOTEBOOK_DIR)

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Graph Orchestrator** (Notebook 5) retrieves it to configure `A2AClientToolProvider` for the recommendation node

In [ ]:
if status.upper() in ["ACTIVE", "READY"]:
    result = store_agent_url(
        param_name=SSM_RECOMMENDATION_AGENT_URL,
        url=recommendation_agent_url,
        region=region,
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

In [ ]:
print("=" * 60)
print("Recommendation Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {RECOMMENDATION_AGENT_NAME}")
print(f"Agent ARN: {RECOMMENDATION_AGENT_ARN}")
print(f"IAM Role: {recommendation_role_arn}")
print(f"Runtime URL: {recommendation_agent_url or 'Not available - agent not ready'}")
print(f"SSM Parameter: {SSM_RECOMMENDATION_AGENT_URL}")
print("=" * 60)

---
## Step 3: Test Recommendation Agent

Verify the agent works standalone before integrating with the Graph Orchestrator. When tested in isolation, the agent generates recommendations from the context provided in the test message (simulating what GraphBuilder would pass from predecessor nodes).

**Note:** In the full graph flow, the Recommendation Agent receives product catalog and order history from upstream nodes automatically. For standalone testing, include sample context in the test message.

In [ ]:
test_message = """Based on the following data, provide personalized recommendations:

Product catalog: laptops ($800-$1500), smartphones ($400-$1200), tablets ($300-$800), headphones ($50-$300)
Customer order history: Previously purchased a MacBook Pro ($1299) and AirPods ($179)

Recommend 3-5 products this customer might like."""

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}],
        }
    },
}

os.chdir(recommendation_agent_dir)
print("Testing Recommendation Agent with sample context...")
print("-" * 40)
response = recommendation_runtime.invoke(payload, session_id=str(uuid4()))
print(f"\nResponse:\n{response}")
os.chdir(NOTEBOOK_DIR)

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **5. Deploy Graph Orchestrator** | GraphBuilder DAG that routes through all 4 agents with conditional edges, parallel execution, and join handling |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
print("Destroying Recommendation Agent...")
os.chdir(recommendation_agent_dir)
try:
    recommendation_runtime.destroy(delete_ecr_repo=True)
    print("Recommendation Agent destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)

In [ ]:
ssm = boto3.client("ssm", region_name=region)
try:
    ssm.delete_parameter(Name=SSM_RECOMMENDATION_AGENT_URL)
    print(f"Deleted SSM parameter: {SSM_RECOMMENDATION_AGENT_URL}")
except Exception as e:
    print(f"Error deleting SSM parameter: {e}")

print("Cleaning up auto-generated files...")
for cleanup_file in ["Dockerfile", ".dockerignore"]:
    cleanup_path = recommendation_agent_dir / cleanup_file
    if cleanup_path.exists():
        cleanup_path.unlink()
        print(f"  Deleted: {cleanup_file}")